In [0]:
BASE_DIR = '/Volumes/sales/landing/volumes/Sellers/'
CHECKPOINT_DIR = '/Volumes/sales/landing/volumes/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/Sellers'
CATALOG = 'SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'SELLERS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
sellers_schema = """seller_id STRING, 
                 seller_zip_code_prefix INT, 
                 seller_city  STRING,
                 seller_state STRING
                 """

In [0]:
sellers_df = (spark.read.format('csv').option('header','true').schema(sellers_schema).load(f'{BASE_DIR}')
           )
display(sellers_df.limit(3))
sellers_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

sellers_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(sellers_schema)
                 .load(f'{BASE_DIR}')
        )
sellers_trans_df = sellers_df.select(
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())

In [0]:
qry = (
    sellers_trans_df.writeStream.format('delta')
    .outputMode('append')
    .option('checkpointLocation', CHECKPOINT_LOCATION)
    .trigger(availableNow=True)
    .toTable(f'{CATALOG}.{SCHEMA}.{TARGET_TABLE}')
)
qry.awaitTermination()